# Statistics Foundations: Distributions & Hypothesis Testing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/01_Fundamentals/statistics_hypothesis_testing.ipynb)

Before models come distributions and tests: is this A/B lift real? Is this feature different between groups? This notebook builds the core toolkit with scipy.

Topics: distributions, CLT simulation, t-tests, chi-squared, p-value pitfalls.

## 1. The distributions you will actually meet

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
x = np.linspace(-5, 5, 400)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
axes[0].plot(x, stats.norm.pdf(x));           axes[0].set_title("Normal")
axes[1].plot(x, stats.expon.pdf(x, scale=1)); axes[1].set_title("Exponential")
axes[2].bar(range(5), stats.poisson.pmf(range(5), mu=2)); axes[2].set_title("Poisson(mu=2)")
axes[3].plot(x, stats.t.pdf(x, df=3), label="t df=3")
axes[3].plot(x, stats.norm.pdf(x), "--", label="normal"); axes[3].legend()
axes[3].set_title("Student-t (fat tails)")
plt.tight_layout(); plt.show()

## 2. Central Limit Theorem by simulation

In [ ]:
rolls = rng.exponential(1, size=(100_000,))            # very non-normal source
means = rolls.reshape(50_000, 10).mean(axis=1)         # averages of n=10
means100 = rolls.reshape(10_000, 100).mean(axis=1)     # averages of n=100

plt.hist(means, bins=60, alpha=.5, density=True, label="mean of 10")
plt.hist(means100, bins=60, alpha=.5, density=True, label="mean of 100")
xs = np.linspace(0, 2, 300)
plt.plot(xs, stats.norm.pdf(xs, 1, 1/np.sqrt(100)), "k--", label="CLT normal")
plt.legend(); plt.title("Sample means become normal regardless of source"); plt.show()

## 3. A/B test: two-proportion z/t test

In [ ]:
visitors_a, clicks_a = 10_000, 420
visitors_b, clicks_b = 10_000, 486

count = np.array([clicks_a, clicks_b])
nobs  = np.array([visitors_a, visitors_b])
z_stat, p_value = stats.proportions_ztest(count, nobs)
lift = clicks_b / visitors_b - clicks_a / visitors_a
print(f"control CTR : {clicks_a/visitors_a:.2%}")
print(f"variant CTR : {clicks_b/visitors_b:.2%}  (lift {lift:.2%})")
print(f"z={z_stat:.3f}, p={p_value:.4f} ->",
      "SIGNIFICANT at 5%" if p_value < 0.05 else "not significant")

## 4. Two-sample t-test + chi-squared

In [ ]:
group1 = rng.normal(50, 10, 40)         # e.g. scores method A
group2 = rng.normal(55, 10, 40)         # method B
t, p = stats.ttest_ind(group2, group1, equal_var=False)
print(f"Welch t-test: t={t:.3f}, p={p:.4f}")

# categorical association: gender vs product preference
table = np.array([[30, 10], [25, 35]])  # rows: M/F, cols: like/dislike
chi2, p_chi, dof, exp = stats.chi2_contingency(table)
print(f"Chi-squared: chi2={chi2:.2f}, p={p_chi:.4f}, dof={dof}")

## P-value discipline (avoid the classic sins)
- p < 0.05 does NOT mean "probability hypothesis true".
- Peek at results repeatedly => inflated false positives (use sequential corrections).
- Significant != important - always report effect size (lift %, Cohen's d).
- Multiple comparisons? Bonferroni/FDR.
- Check assumptions: normality (Shapiro), equal variance (Levene) before t-tests.